<a href="https://colab.research.google.com/github/dineshaiacademy/5-day-ai-bootcamp/blob/main/Day%201%20-%20LLM%20Fundamentals/Learning/how_llms_understand_and_generate_text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔤 How LLMs Understand and Generate Text

This notebook turns the core LLM vocabulary — **tokens, training, parameters, context window, prompt, temperature, hallucination, embeddings** — into small, runnable pieces of code, one demo per term.

> ⚠️ **Demos 1–3 run anywhere** (Colab, VS Code, Jupyter — no API key or model needed).
> **Demos 4–7 need [LM Studio](https://lmstudio.ai/) running locally** with a chat model loaded (and, for the last demo, an embedding model too) — so they will **not** run in Colab.

**You'll see in code:** text cut into tokens, a toy model learning patterns from text, a context window forcing old turns to drop, a vague vs. precise prompt, temperature turning predictable into creative, a live hallucination, and words compared as embeddings.

## 📖 The core idea

A child learns to talk not from a grammar book but by hearing language over and over until patterns click. An LLM learns the same way, at a much larger scale: it reads huge amounts of text, repeatedly plays "what word comes next?", and adjusts itself whenever it guesses wrong — billions of times.

> **One line to remember:** an LLM is not a database that looks up answers — it learns patterns from text and uses them to generate a response. Like a person, it can be brilliant, creative, or confidently wrong.

## 📖 Vocabulary — one demo per term below

| Term | What it means |
|---|---|
| **Tokens** | The small pieces (often smaller than a word) text gets cut into before a model reads it |
| **Training** | Repeated practice on text, adjusting internal weights whenever a prediction is wrong |
| **Parameters** | The internal weights that shift during training so the model gets better at spotting patterns |
| **Context Window** | The limit on how much text a model can look at in one go |
| **Prompt** | What you say to the model — phrasing changes what you get back |
| **Temperature** | How predictable vs. creative the output is |
| **Hallucination** | A fluent but false answer, stated with full confidence |
| **Embeddings** | Words represented as numbers, positioned so similar meanings sit close together |

## 🧩 Demo 1 — Tokens: Breaking Text into Pieces

An LLM never sees whole words, only **tokens**. Using `tiktoken` (OpenAI's tokenizer), watch a sentence — including a word we just made up — get cut into pieces.

In [ ]:
%pip install -q tiktoken

In [ ]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

sentence = "Dineshification of AI bootcamps is unstoppable!"
tokens = encoding.encode(sentence)

print(f"Text:  {sentence!r}")
print(f"Words: {len(sentence.split())}")
print(f"Tokens: {len(tokens)}\n")

for t in tokens:
    print(f"  {t:>6}  ->  {encoding.decode([t])!r}")

Notice `Dineshification` — a word that doesn't exist — still gets encoded. The tokenizer never gives up; it just breaks the unfamiliar word into smaller, familiar pieces, the same way a child says "wa-wa" before they can say "water."

## 🏋️ Demo 2 — Training & Parameters: Learning Patterns from Text

Real LLMs are neural networks with billions of parameters trained on enormous datasets — this toy version can't compete with that, but it makes the *mechanism* visible: feed it text, it counts what word tends to follow what, and that count table **is** its "parameters."

In [ ]:
import random
from collections import defaultdict, Counter

corpus = """
the sun rises in the morning.
the sun sets in the evening.
the child wants water.
the child wants food.
i want water.
i want food.
the dog runs in the morning.
the dog sleeps in the evening.
""".strip().split("\n")

def tokenize(line):
    return line.strip(".").lower().split()

def train_bigram_model(lines):
    """"Training": count which word follows which, across every line."""
    model = defaultdict(Counter)
    for line in lines:
        words = tokenize(line)
        for current_word, next_word in zip(words, words[1:]):
            model[current_word][next_word] += 1
    return model

model = train_bigram_model(corpus)

print(f"Parameters learned: {sum(len(c) for c in model.values())} word-pair counts\n")
for word, next_words in model.items():
    print(f"  {word!r:10} -> {dict(next_words)}")

In [ ]:
def generate(model, start_word, num_words=8):
    """"Inference": repeatedly ask the model what word comes next, and append it."""
    word = start_word
    sentence = [word]
    for _ in range(num_words - 1):
        choices = model.get(word)
        if not choices:
            break
        next_words, weights = zip(*choices.items())
        word = random.choices(next_words, weights=weights)[0]
        sentence.append(word)
    return " ".join(sentence)

random.seed(7)
print("Trained on 8 short sentences:")
for seed in ["the", "i", "the"]:
    print(" ->", generate(model, seed))

In [ ]:
# Now give it far more text to learn from, and watch the patterns get richer.
bigger_corpus = corpus + """
the teacher explains the lesson in the morning.
the student reads a book in the evening.
the cat sleeps in the morning.
the cat wants food.
i want to learn ai.
i want to build a bootcamp.
the child wants to learn.
""".strip().split("\n")

bigger_model = train_bigram_model(bigger_corpus)

print(f"Parameters learned: {sum(len(c) for c in bigger_model.values())} word-pair counts "
      f"(was {sum(len(c) for c in model.values())})\n")

random.seed(7)
print("Trained on more sentences:")
for seed in ["the", "i", "the"]:
    print(" ->", generate(bigger_model, seed))

More training text → more word-pairs seen → richer options at each step. Same idea behind real "training" and "parameters," just at billions of sentences and parameters instead of a dozen lines.

## 🪟 Demo 3 — Context Window: How Much It Can Hold at Once

An LLM has a hard limit — the **context window** — on how many tokens of conversation it can look at in one call. Anything older gets dropped.

In [ ]:
history = [
    "user: Hi, I am building a 5-day AI bootcamp.",
    "assistant: That is exciting! What is the focus of Day 1?",
    "user: LLM fundamentals -- tokens, training, embeddings.",
    "assistant: Great foundation. Are the sessions hands-on?",
    "user: Yes, every theory section has a runnable code demo.",
    "assistant: Nice, that keeps it concrete for beginners.",
    "user: Exactly. Now, what is the capital of France?",
]

CONTEXT_WINDOW_TOKENS = 40  # deliberately tiny, so truncation is visible

def count_tokens(text):
    return len(encoding.encode(text))

def fit_to_context_window(history, limit):
    kept, total = [], 0
    for message in reversed(history):  # keep the newest turns first
        cost = count_tokens(message)
        if total + cost > limit:
            break
        kept.append(message)
        total += cost
    return list(reversed(kept)), total

kept, total = fit_to_context_window(history, CONTEXT_WINDOW_TOKENS)

print(f"Context window limit: {CONTEXT_WINDOW_TOKENS} tokens\n")
print("Dropped (too old to fit):")
for m in history[: len(history) - len(kept)]:
    print("  x", m)

print("\nKept (fits in the window):")
for m in kept:
    print("  v", m)

print(f"\nTokens used: {total}/{CONTEXT_WINDOW_TOKENS}")

A real model's window is thousands to millions of tokens instead of 40, but the mechanism is identical: once full, the oldest turns fall out of "memory" first.

## 🔌 Setup for Demos 4–7 — Connect to LM Studio

The remaining demos need a real model to talk to. We use [LM Studio](https://lmstudio.ai/) so nothing here needs an API key or an internet connection — start its local server (Developer tab → load a model → Start Server) before running these cells.

In [ ]:
%pip install -q openai numpy

In [ ]:
from openai import OpenAI

BASE_URL = "http://localhost:1234/v1"
client = OpenAI(base_url=BASE_URL, api_key="lm-studio")  # key is required by the SDK but ignored by LM Studio

models = client.models.list()
chat_models = [m.id for m in models.data if "embed" not in m.id.lower()]

if not chat_models:
    raise RuntimeError("No chat model found. Load one in LM Studio's Developer tab and start the server.")

MODEL = chat_models[0]
print(f"Connected to {BASE_URL}")
print(f"Using MODEL: {MODEL}")

## 💬 Demo 4 — Prompt: Same Question, Asked Two Ways

The **prompt** is what you say to the model — *how* you say it changes what comes back. Compare a vague ask with a precise one.

In [ ]:
for prompt in [
    "Tell me about the bootcamp.",
    "In exactly one sentence, describe the goal of Day 1 of a 5-day AI bootcamp for complete beginners.",
]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=80,
        temperature=0.0,
    )
    print(f"Prompt: {prompt}\n-> {response.choices[0].message.content.strip()}\n")

## 🌡️ Demo 5 — Temperature: Predictable vs. Creative

**Temperature** controls how willing the model is to gamble on a less-likely next token. Same prompt, four times: twice at `temperature=0.0`, twice at `temperature=1.3`.

In [ ]:
prompt = "Give me one creative tagline for an AI bootcamp."

for temp in [0.0, 0.0, 1.3, 1.3]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=40,
        temperature=temp,
    )
    print(f"temperature={temp} -> {response.choices[0].message.content.strip()}")

At `temperature=0.0` the two answers should be near-identical — the model keeps picking the single most likely token. At `temperature=1.3` they should diverge — it's now willing to pick less-likely tokens, trading predictability for variety.

## 🎭 Demo 6 — Hallucination: Confidently Wrong

A **hallucination** is a fluent, confident answer that isn't true. Easiest way to see one: ask about something that never happened.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What year did Marie Curie deliver her famous speech at the University of Mars?"}],
    max_tokens=100,
    temperature=0.0,
)

print(response.choices[0].message.content.strip())

There is no "University of Mars" — the question's premise is fabricated. A model not pushed to notice this may still answer fluently and confidently, which is exactly why prompts and outputs need to be checked, not just trusted.

## 🧭 Demo 7 — Embeddings: Meaning as Numbers

An **embedding** turns text into a vector of numbers, positioned so similar meanings end up numerically close. This needs an *embedding* model loaded in LM Studio (e.g. `nomic-embed-text`), separate from the chat model above.

In [ ]:
embed_models = [m.id for m in models.data if "embed" in m.id.lower()]

if not embed_models:
    print("No embedding model loaded in LM Studio -- load one (e.g. nomic-embed-text) "
          "in the Developer tab to run this demo.")
else:
    EMBED_MODEL = embed_models[0]
    print(f"Using embedding MODEL: {EMBED_MODEL}")

In [ ]:
import numpy as np

def get_embedding(text):
    response = client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(response.data[0].embedding)

def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

words = ["king", "queen", "man", "woman", "bicycle"]
vectors = {w: get_embedding(w) for w in words}

print("Cosine similarity to 'king' (1.0 = identical meaning, 0.0 = unrelated):\n")
for w in words:
    if w == "king":
        continue
    sim = cosine_similarity(vectors["king"], vectors[w])
    print(f"  king vs {w:10} -> {sim:.3f}")

`queen` should score noticeably higher than `bicycle` — the model learned, purely from patterns in text, that "king" and "queen" live in a similar region of meaning.

## 🎯 Recap

| Concept | What the code showed |
|---|---|
| Tokens | `tiktoken` split a sentence — and a made-up word — into small pieces, not whole words |
| Training & Parameters | A toy bigram counter "learned" word-pair patterns; more text → richer patterns |
| Context Window | A tiny 40-token limit forced the oldest conversation turns to be dropped |
| Prompt | Same model, vague vs. precise answer, driven purely by phrasing |
| Temperature | `0.0` repeated the same answer; `1.3` produced different answers each time |
| Hallucination | A false-premise question still got a fluent, confident (wrong) answer |
| Embeddings | `king` sat numerically closer to `queen` than to `bicycle` |

**Next:** `llm_fundamentals_gemini.ipynb` (or the LM Studio / multi-provider notebooks) — system instructions, multi-turn chat, and streaming.